# DataForge Phase 4: GCP read-only audit
This notebook validates **real BigQuery metadata and dry runs only** in *your* authenticated Google Cloud project. It does not modify DataPulse or execute queries by default. Pub/Sub checks read metadata only. Cloud resources may be missing.


In [ ]:
!git clone -q https://github.com/Codewithadi21/dataforge-ai.git
%cd dataforge-ai
!pip -q install -e ".[cloud]"


## Google authentication
Run the following cell, complete the sign-in via Google Colab, then enter the Google Cloud project ID that owns your DataPulse resources. **Never paste keys or service-account JSON into this notebook.**


In [ ]:
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery
GCP_PROJECT_ID = "REPLACE_WITH_YOUR_GCP_PROJECT_ID"  # project ID only, not a secret
assert GCP_PROJECT_ID != "REPLACE_WITH_YOUR_GCP_PROJECT_ID"
client = bigquery.Client(project=GCP_PROJECT_ID)


## BigQuery: metadata + dry runs (default)
Checks baseline_customers, shifted_customers and paired_prediction_replay in datapulse_ai, using COUNT(*) dry-run compilation with no data read. No insert, delete, load, or model operation.


In [ ]:
import json
from dataforge.cloud_validation import audit_bigquery_dry_runs
audit = audit_bigquery_dry_runs(client, GCP_PROJECT_ID, max_bytes=100_000_000, location="US")
print(json.dumps(audit, indent=2))


## Pub/Sub: topology metadata only
Checks whether the topic `datapulse-incidents` exists and the subscription `datapulse-incidents-consumer` points to it. No publishing, pulling or acking.


In [ ]:
from dataforge.cloud_validation import audit_pubsub_topology
try:
    pubsub_result = audit_pubsub_topology(GCP_PROJECT_ID)
    print(json.dumps(pubsub_result, indent=2))
except Exception as exc:
    print("Pub/Sub metadata check unavailable:", type(exc).__name__)


## OPTIONAL billable aggregate — explicit approval only
This cell is **off** by default. Setting APPROVE_BILLABLE_READ_ONLY_QUERY=True executes one capped SELECT aggregate query on the synthetic paired replay table. It may consume trial credits. A GCP budget alert is not a hard cap.


In [ ]:
from dataforge.cloud_validation import audit_opt_in_aggregates
APPROVE_BILLABLE_READ_ONLY_QUERY = False  # manually change to True if you consent
result = audit_opt_in_aggregates(client, GCP_PROJECT_ID, approved_by_user=APPROVE_BILLABLE_READ_ONLY_QUERY, max_bytes=100_000_000, location="US")
print(json.dumps(result, indent=2))


## Final status
Capture these outputs *without credentials*. A successful metadata/dry-run check confirms schema access only; it does not independently verify row contents, production data freshness, Pub/Sub event delivery, or Qwen availability. To demonstrate live inference, separately configure the Streamlit Hugging Face token and verify `Backend: qwen-hosted`.
